In [15]:
%load_ext autoreload
%autoreload 2

In [16]:
import kagglehub

path = kagglehub.dataset_download("hojjatk/mnist-dataset")

In [11]:
import os

print(os.listdir(path))
print("\n")
print(os.listdir(path + '/train-images-idx3-ubyte/'))

['t10k-images-idx3-ubyte', 't10k-images.idx3-ubyte', 't10k-labels-idx1-ubyte', 't10k-labels.idx1-ubyte', 'train-images-idx3-ubyte', 'train-images.idx3-ubyte', 'train-labels-idx1-ubyte', 'train-labels.idx1-ubyte']


['train-images-idx3-ubyte']


In [12]:
import numpy as np

def load_images(path):
    with open(path, 'rb') as f:
        magic, n, rows, cols = np.frombuffer(f.read(16), dtype='>i4')
        images = np.frombuffer(f.read(), dtype=np.uint8)
    return images.reshape(n, rows, cols)

def load_labels(path):
    with open(path, 'rb') as f:
        magic, n = np.frombuffer(f.read(8), dtype='>i4')
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return labels

X_train = load_images(path + '/train-images-idx3-ubyte/train-images-idx3-ubyte')
y_train = load_labels(path + '/train-labels-idx1-ubyte/train-labels-idx1-ubyte')
X_test  = load_images(path + '/t10k-images-idx3-ubyte/t10k-images-idx3-ubyte')
y_test  = load_labels(path + '/t10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte')

In [17]:
from sklearn import preprocessing

scaler = preprocessing.StandardScaler()
def process_x(X, fit):
    N, H, W = X.shape
    X_flat = X.reshape(N, H * W)
    if fit:
        scaler.fit(X_flat)
    return scaler.transform(X_flat)

X_train = process_x(X_train, True)
X_test = process_x(X_test, False)

In [19]:
# Show a few
from matplotlib import pyplot as plt
from ipywidgets import interact, IntSlider

interact(
    lambda i: plt.imshow(X_train[i].reshape(28, 28), cmap="gray"), i=IntSlider(min=0, max=len(y_train) - 1)
)

interactive(children=(IntSlider(value=0, description='i', max=59999), Output()), _dom_classes=('widget-interac…

<function __main__.<lambda>(i)>

In [20]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def make_loader(X, y, batch_size=256, shuffle=False):
    dataset = TensorDataset(
        torch.from_numpy(X.astype(np.float32)),
        torch.from_numpy(y),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=4)

train_loader = make_loader(X_train, y_train, shuffle=True)
test_loader = make_loader(X_test, y_test)

/tmp/ipykernel_89175/1608567004.py:7: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  torch.from_numpy(y),


In [22]:
from SparseCAE import SparseCAE
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping

model = SparseCAE(
    input_dim=28 * 28,
    latent_dim=16,
    hidden_dims=[256],
    lambda_sparse=1e-3,
    lambda_cae=1e-4,
    lr=1e-3,
)

trainer = pl.Trainer(
    max_epochs=50,
    accelerator="auto",
    callbacks=[
        EarlyStopping(monitor="val/loss", patience=10),
    ],
)
trainer.fit(model, train_loader, test_loader)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name    | Type       | Params | Mode  | FLOPs
-------------------------------------------------------
0 | encoder | Sequential | 205 K  | train | 0    
1 | decoder | Sequential | 205 K  | train | 0    
-------------------------------------------------------
410 K     Trainable params
0         Non-trainable params
410 K     Total params
1.644     Total estimated model params size (MB)
10        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/pipev/Documents/Proyectos/ML-Portfolio/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
